# Fig. 4D, 4E: within-family DRIMSeq (paired), LOPO expression filter

Paired DRIMSeq `~ family + Diagnosis`, genewise precision, gene-level results, with the tighter
LOPO expression filter (an independent-filtering refinement). An earlier version with a lenient
expression filter and otherwise the same method gave 8 genes (panel output archived at
`fig_s5h_within_family_drimseq__lenient8_ARCHIVE/`).

Result: 20 genes at FDR<0.05, 33 at FDR<0.1 (8 / 20 with the lenient filter; the lenient count
is reproduced by the `very_lenient` sweep row, which confirms that the method matches). Most hits
lengthen the 3'UTR (fraction in panel b), the same direction as the aggregate within-family shift
(`analysis/blood/07-260615-within_family_systematic_apa.ipynb`) and the post-mortem cortex (Fig. 3).
Functional groups: immune/myeloid CTSS/DOCK8/MS4A6A/CXCR2/PIM1/GIMAP5/LST1/STAT1; RNA-binding/
poly(A) PABPC1/YBX1/YBX3/ZFP36L1/HNRNPC/RPS23; GTPase/cytoskeleton CYTH1/GRB2/RIPOR2/KIF13B/
CAPZA2/VIM.

The tighter filter is an independent filter (it depends on expression only, not on diagnosis),
so it does not inflate false positives; it recovers more genes by lowering the BH
multiple-testing burden and de-noising the per-gene DM fits.

Caveat: n=9 families, ~8 residual df, so the LRT FDR is uncertain (mildly liberal); the hits are
supported as a group (direction, biology, agreement across designs) rather than as individual
per-gene calls.

Panel name: `fig_s5h_within_family_drimseq`. DRIMSeq run:
`analysis/blood/05-260714-DRIMSeq_all_sibling_pairs_LOPOfilter.R`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys
import warnings

warnings.filterwarnings("ignore")
import anndata as ad
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from scipy.stats import rankdata, binomtest

sys.path.insert(0, f"{REPO_ROOT}/figures/style")
import paper_fig_style as S

S.apply_rcparams()
PANEL = "fig_s5h_within_family_drimseq"  # S5h panel name (LOPO-filter content)
GL = f"{DATA_ROOT}/3utr/Blood/all_till_20240724/utrcalling/UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"
ALLDIR = f"{DATA_ROOT}/3utr/Blood/2025_analyses/DRIMSeq_allpairs_genewise/"
LOPODIR = f"{DATA_ROOT}/3utr/Blood/2025_analyses/LOPO_siblings_common/"

a = ad.read_h5ad(GL)
X = a.X
X = np.asarray(X.todense()) if hasattr(X, "todense") else np.asarray(X)
gene = a.var["name"].astype(str).values
length = pd.to_numeric(a.var["utr_size"], errors="coerce").values.astype(float)
o = a.obs.copy()
o["age"] = pd.to_numeric(o.age_specific, errors="coerce")
o["Dx"] = o.Diagnosis.astype(str)
o["fam"] = o.family.astype(str)
o["fm"] = o.family_member.astype(str)
o["Bt"] = o.Batch.astype(str)
samp = o.index.values

# within-family mean Delta-psi per gene (volcano x-axis; same values as the lenient-filter version)
r = np.zeros(len(gene))
df = pd.DataFrame({"g": gene, "len": length})
for g, idx in df.groupby("g").groups.items():
    ii = np.array(list(idx))
    K = len(ii)
    r[ii] = 0.0 if K < 2 else (rankdata(length[ii], method="average") - 1) / (K - 1)
num = pd.DataFrame((X * r[None, :]).T, index=gene).groupby(level=0).sum()
num.columns = samp
den = pd.DataFrame(X.T, index=gene).groupby(level=0).sum()
den.columns = samp
W = num / den.replace(0, np.nan)
pairs = []
for (b, f), g in o.groupby(["Bt", "fam"]):
    asd = g[g.Dx == "ASD"]
    ctl = g[(g.Dx == "CTRL") & (g.fm.isin(["Sibling", "Child"]))]
    if len(asd) and len(ctl):
        for _, pr in asd.iterrows():
            cc = ctl.loc[(ctl.age - pr.age).abs().idxmin()]
            pairs.append((pr.name, cc.name))
P = pd.DataFrame(pairs, columns=["proband", "sib"])
pro, sib = P.proband.values, P.sib.values
Dp = W[pro].values - W[sib].values
EV = (den[pro].values >= 10) & (den[sib].values >= 10) & ~np.isnan(Dp)
mdpsi = pd.Series(
    np.where(
        EV.sum(1) >= 1,
        np.nansum(np.where(EV, Dp, 0), 1) / np.maximum(EV.sum(1), 1),
        np.nan,
    ),
    index=W.index,
)

# --- LOPO-filter DRIMSeq gene-level result ---
dr = pd.read_csv(
    ALLDIR + "drimseq_allpairs_genelevel_LOPOfilter.csv"
)  # gene_id, symbol, lr, df, pvalue, adj_pvalue
SYM_FIX = {"GIMAP1-GIMAP5|GIMAP5": "GIMAP5"}  # tidy the one comma-joined multi-gene locus
dr["symbol"] = dr.symbol.astype(str).map(lambda s: SYM_FIX.get(s, s))
dr["dpsi"] = dr.gene_id.map(mdpsi)
dr["nl"] = -np.log10(dr.adj_pvalue.clip(lower=1e-300))
dr["nlp"] = -np.log10(dr.pvalue.clip(lower=1e-300))  # Nominal p (volcano y-axis)
dr["t05"] = dr.adj_pvalue < 0.05
dr["t10"] = dr.adj_pvalue < 0.1

# --- LOPO fold-count per gene (top-50 LOPO classifier, supp_9): #distinct folds a gene's peak was selected in ---
fsel = pd.read_csv(LOPODIR + "lopo_feat_sel.csv")
stab = pd.read_csv(LOPODIR + "lopo_feature_stability.csv")
f2g = dict(zip(stab.feature_id, stab.gene_id))
fsel["gene_id"] = fsel.feature_id.map(f2g)
gene_folds = fsel.groupby("gene_id")["fold"].nunique().to_dict()
dr["lopo_folds"] = dr.gene_id.map(lambda g: int(gene_folds.get(g, 0)))

H = dr[dr.t10].dropna(subset=["dpsi"]).copy()
THEME = {
    "CTSS": "immune/myeloid",
    "DOCK8": "immune/myeloid",
    "MS4A6A": "immune/myeloid",
    "CXCR2": "immune/myeloid",
    "PIM1": "immune/myeloid",
    "GIMAP5": "immune/myeloid",
    "LST1": "immune/myeloid",
    "STAT1": "immune/myeloid",
    "PABPC1": "RNA-binding/poly(A)",
    "YBX1": "RNA-binding/poly(A)",
    "ZFP36L1": "RNA-binding/poly(A)",
    "RPS23": "RNA-binding/poly(A)",
    "YBX3": "RNA-binding/poly(A)",
    "HNRNPC": "RNA-binding/poly(A)",
    "CYTH1": "GTPase/cytoskeleton",
    "GRB2": "GTPase/cytoskeleton",
    "PTPRA": "GTPase/cytoskeleton",
    "RIPOR2": "GTPase/cytoskeleton",
    "KIF13B": "GTPase/cytoskeleton",
    "CAPZA2": "GTPase/cytoskeleton",
    "VIM": "GTPase/cytoskeleton",
    "H3-3B": "chromatin",
    "ZNF706": "chromatin",
    "TCF25": "chromatin",
    "SELENOK": "other",
    "UQCRB": "other",
    "ADIPOR1": "other",
    "ST13": "other",
    "CYSTM1": "other",
    "SNCA": "other",
    "STEAP4": "other",
    "GNPTG": "other",
    "TXNIP": "other",
    "APEX1": "other",
}
H["theme"] = H.symbol.map(lambda s: THEME.get(str(s).split("/")[0], "other"))
nlen = int((H.dpsi > 0).sum())
ntot = len(H)
bp = binomtest(nlen, ntot).pvalue
print(
    "LOPO-filter within-family DTU | FDR<0.1:",
    len(H),
    "| FDR<0.05:",
    int(H.t05.sum()),
    "| lengthening %d/%d (p=%.4f)" % (nlen, ntot, bp),
)

In [ ]:
CATCOLS = ["immune/myeloid", "RNA-binding/poly(A)", "GTPase/cytoskeleton", "chromatin"]
CC = {
    "immune/myeloid": "#b2182b",
    "RNA-binding/poly(A)": "#762a83",
    "GTPase/cytoskeleton": "#2166ac",
    "chromatin": "#1a9850",
}
CLAB = {
    "immune/myeloid": "immune",
    "RNA-binding/poly(A)": "RNA/polyA",
    "GTPase/cytoskeleton": "GTPase/cyto",
    "chromatin": "chromatin",
}
catmemb = pd.read_csv(ALLDIR + "gene_category_curated.csv").set_index("symbol")
S20 = dr[dr.t05].dropna(subset=["dpsi"]).copy().sort_values("dpsi")
nlen20 = int((S20.dpsi > 0).sum())
ntot20 = len(S20)
figh_mm = max(70, 3.6 * ntot20 + 22)
fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(figh_mm) / 300))
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 0.8, 0.5], wspace=0.30)
# (a) within-family DRIMSeq volcano - nominal -log10 p on y; colour by corrected significance (adj<0.05)
ax = fig.add_subplot(gs[0, 0])
ns = dr[~dr.t05].dropna(subset=["dpsi", "nlp"])
sig = dr[dr.t05].dropna(subset=["dpsi", "nlp"])
ax.scatter(ns.dpsi, ns.nlp, s=5, c="#cccccc", alpha=0.5, edgecolors="none", rasterized=True)
ax.scatter(
    sig.dpsi,
    sig.nlp,
    s=22,
    c="#2c7fb8",
    edgecolors="k",
    linewidths=0.3,
    label="adj-p < 0.05",
)
ax.axvline(0, ls=":", c="grey", lw=0.5)  # lengthen/shorten divider (not a significance threshold)
try:
    from adjustText import adjust_text

    t = [
        ax.text(r.dpsi, r.nlp, r.symbol, fontsize=S.FONT_ANNOT - 0.5, fontstyle="italic")
        for _, r in sig.sort_values("pvalue").head(12).iterrows()
    ]
    adjust_text(t, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
except Exception:
    pass
ax.set_xlabel(r"within-family $\Delta\psi$ (ASD - sib; + = lengthening)", fontsize=S.FONT_LABEL)
ax.set_ylabel(r"$-\log_{10}$ p-value (nominal; paired DRIMSeq)", fontsize=S.FONT_LABEL)
ax.legend(fontsize=S.FONT_LEGEND, frameon=False, loc="upper left")
ax.set_title("(a) within-family DTU (paired ~family+diagnosis)", fontsize=S.FONT_ANNOT)
ax.tick_params(labelsize=S.FONT_TICK)
[s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]
# (b) dpsi lollipop for the FDR<0.05 genes (neutral dots; category shown in panel c)
axb = fig.add_subplot(gs[0, 1])
yy = np.arange(ntot20)
for y, (_, rr) in zip(yy, S20.iterrows()):
    axb.plot([0, rr.dpsi], [y, y], color="#b0b0b0", lw=1.0, zorder=1)
    axb.scatter(rr.dpsi, y, s=24, color="#333333", zorder=2, edgecolors="none")
axb.axvline(0, c="grey", lw=0.6)
axb.set_ylim(-0.7, ntot20 - 0.3)
axb.set_yticks(yy)
axb.set_yticklabels(["$\\it{%s}$" % s for s in S20.symbol], fontsize=S.FONT_TICK - 0.5)
axb.set_xlabel(r"within-family $\Delta\psi$ (ASD - sib)", fontsize=S.FONT_LABEL)
axb.tick_params(labelsize=S.FONT_TICK, length=0)
for sp in ("top", "right", "left"):
    axb.spines[sp].set_visible(False)
axb.set_title("(b) FDR<0.05: %d/%d lengthen" % (nlen20, ntot20), fontsize=S.FONT_ANNOT)
# (c) GO-curated functional-category membership matrix (multi-membership; gene_category_curated.csv)
axc = fig.add_subplot(gs[0, 2], sharey=axb)
for j, cn in enumerate(CATCOLS):
    for y, (_, rr) in zip(yy, S20.iterrows()):
        member = bool(catmemb.loc[rr.symbol, cn]) if rr.symbol in catmemb.index else False
        if member:
            axc.scatter(j, y, s=44, color=CC[cn], edgecolors="k", linewidths=0.3, zorder=3)
        else:
            axc.scatter(j, y, s=6, color="#e8e8e8", edgecolors="none", zorder=1)
axc.set_xticks(range(len(CATCOLS)))
axc.set_xticklabels([CLAB[c] for c in CATCOLS], rotation=45, ha="right", fontsize=S.FONT_TICK - 0.5)
for tl, cn in zip(axc.get_xticklabels(), CATCOLS):
    tl.set_color(CC[cn])
axc.set_xlim(-0.6, len(CATCOLS) - 0.4)
axc.tick_params(labelleft=False, length=0)
for sp in ("top", "right", "left", "bottom"):
    axc.spines[sp].set_visible(False)
axc.set_title("(c) function (GO-curated)", fontsize=S.FONT_ANNOT)
fig.subplots_adjust(left=0.07, right=0.99, bottom=max(0.07, 0.12 * 70 / figh_mm), top=0.93)
S.save_and_show(fig, PANEL, figure="Fig5/supp_8")

## Quarter-height version (a/b/c prefixes removed; middle y-labels auto-shrunk to fit)

Same 3-subpanel content at quarter height (`FULL_W x QUARTER_H`). The middle lollipop's gene
y-labels are shrunk (font size derived from the panel height / #genes) so the 20 labels do not
overlap, and the "(a)/(b)/(c)" prefixes are dropped from the subpanel titles. Saved as
`fig_s5h_within_family_drimseq__quarter`.

In [ ]:
# --- Quarter-height version of the S5h panel (no (a)/(b)/(c) prefixes; y-labels shrunk to fit) ---
# Reuses S20, nlen20, ntot20, CATCOLS, CC, CLAB, catmemb, dr computed above.
figh_mm = S.QUARTER_H_MM
fig = plt.figure(figsize=(S.FULL_W_PX / 300, S.mm_to_px(figh_mm) / 300))
gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 0.8, 0.5], wspace=0.30)
_top, _bottom = 0.87, 0.20
_plot_h_mm = (_top - _bottom) * figh_mm
fs_b = float(np.clip(2.1 * _plot_h_mm / ntot20, 3.5, S.FONT_TICK - 0.5))  # shrink to avoid overlap

# volcano (nominal -log10 p; colour by corrected significance)
ax = fig.add_subplot(gs[0, 0])
_ns = dr[~dr.t05].dropna(subset=["dpsi", "nlp"])
_sig = dr[dr.t05].dropna(subset=["dpsi", "nlp"])
ax.scatter(_ns.dpsi, _ns.nlp, s=4, c="#cccccc", alpha=0.5, edgecolors="none", rasterized=True)
ax.scatter(
    _sig.dpsi, _sig.nlp, s=16, c="#2c7fb8", edgecolors="k", linewidths=0.3, label="adj-p < 0.05"
)
ax.axvline(0, ls=":", c="grey", lw=0.5)
try:
    from adjustText import adjust_text

    _t = [
        ax.text(r.dpsi, r.nlp, r.symbol, fontsize=S.FONT_ANNOT - 1, fontstyle="italic")
        for _, r in _sig.sort_values("pvalue").head(6).iterrows()
    ]
    adjust_text(_t, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
except Exception:
    pass
ax.set_xlabel(r"within-family $\Delta\psi$ (ASD - sib; + = lengthening)", fontsize=S.FONT_TICK)
ax.set_ylabel(r"$-\log_{10}$ p-value (nominal; paired DRIMSeq)", fontsize=S.FONT_TICK)
ax.legend(fontsize=S.FONT_LEGEND - 1, frameon=False, loc="upper left")
ax.set_title("within-family DTU (paired ~family+diagnosis)", fontsize=S.FONT_ANNOT)
ax.tick_params(labelsize=S.FONT_TICK - 1)
[s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]

# lollipop (dpsi for the FDR<0.05 genes)
axb = fig.add_subplot(gs[0, 1])
yy = np.arange(ntot20)
for y, (_, rr) in zip(yy, S20.iterrows()):
    axb.plot([0, rr.dpsi], [y, y], color="#b0b0b0", lw=0.8, zorder=1)
    axb.scatter(rr.dpsi, y, s=11, color="#333333", zorder=2, edgecolors="none")
axb.axvline(0, c="grey", lw=0.6)
axb.set_ylim(-0.7, ntot20 - 0.3)
axb.set_yticks(yy)
axb.set_yticklabels(["$\\it{%s}$" % s for s in S20.symbol], fontsize=fs_b)
axb.set_xlabel(r"within-family $\Delta\psi$ (ASD - sib)", fontsize=S.FONT_TICK)
axb.tick_params(labelsize=S.FONT_TICK, length=0)
for sp in ("top", "right", "left"):
    axb.spines[sp].set_visible(False)
axb.set_title("FDR<0.05: %d/%d lengthen" % (nlen20, ntot20), fontsize=S.FONT_ANNOT)

# GO-curated functional-category membership matrix
axc = fig.add_subplot(gs[0, 2], sharey=axb)
for j, cn in enumerate(CATCOLS):
    for y, (_, rr) in zip(yy, S20.iterrows()):
        member = bool(catmemb.loc[rr.symbol, cn]) if rr.symbol in catmemb.index else False
        if member:
            axc.scatter(j, y, s=16, color=CC[cn], edgecolors="k", linewidths=0.25, zorder=3)
        else:
            axc.scatter(j, y, s=3, color="#e8e8e8", edgecolors="none", zorder=1)
axc.set_xticks(range(len(CATCOLS)))
axc.set_xticklabels([CLAB[c] for c in CATCOLS], rotation=45, ha="right", fontsize=S.FONT_TICK - 1)
for tl, cn in zip(axc.get_xticklabels(), CATCOLS):
    tl.set_color(CC[cn])
axc.set_xlim(-0.6, len(CATCOLS) - 0.4)
axc.tick_params(labelleft=False, length=0)
for sp in ("top", "right", "left", "bottom"):
    axc.spines[sp].set_visible(False)
axc.set_title("function (GO-curated)", fontsize=S.FONT_ANNOT)

fig.subplots_adjust(left=0.07, right=0.99, bottom=_bottom, top=_top)
S.save_and_show(fig, "fig_s5h_within_family_drimseq__quarter", figure="Fig5/supp_8")

## Exploratory volcano - significant genes coloured by LOPO fold-consensus (not a paper panel)

Same within-family DTU volcano, but the adj-p<0.05 genes are coloured by how many of the 9 LOPO
folds selected them (0 = never picked by the classifier; 9 = every fold). This gives a direct
read of gene-level DTU significance / effect size vs the classifier's fold-consensus: the top-left/
top-right strong-effect genes that are also high-fold-count (yellow) are the most consistent within-
family signal; dark-purple significant genes are DTU hits the classifier's top-50 did not use.


In [ ]:
from matplotlib.lines import Line2D

sig = dr[dr.t05].dropna(subset=["dpsi"]).copy()
nsg = dr[~dr.t05].dropna(subset=["dpsi"])
fig, ax = plt.subplots(figsize=(S.mm_to_px(110) / 300, S.mm_to_px(95) / 300))
ax.scatter(
    nsg.dpsi,
    nsg.nl,
    s=6,
    c="#dddddd",
    alpha=0.6,
    edgecolors="none",
    rasterized=True,
    label="ns (adj>=0.05)",
)
sc = ax.scatter(
    sig.dpsi,
    sig.nl,
    c=sig.lopo_folds,
    cmap="viridis",
    vmin=0,
    vmax=9,
    s=48,
    edgecolors="k",
    linewidths=0.4,
    zorder=3,
)
cb = fig.colorbar(sc, ax=ax, ticks=range(0, 10), fraction=0.046, pad=0.02)
cb.set_label("# of 9 LOPO folds selected in", fontsize=8)
cb.ax.tick_params(labelsize=7)
ax.axhline(-np.log10(0.05), ls="--", c="grey", lw=0.6)
ax.axvline(0, ls=":", c="grey", lw=0.6)
try:
    from adjustText import adjust_text

    t = [ax.text(r.dpsi, r.nl, r.symbol, fontsize=7, fontstyle="italic") for _, r in sig.iterrows()]
    adjust_text(t, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
except Exception:
    pass
ax.set_xlabel("within-family dpsi (ASD - sib; + = lengthening)", fontsize=9)
ax.set_ylabel("-log10 adj-p (paired DRIMSeq, LOPO filter)", fontsize=9)
ax.set_title("within-family DTU (adj<0.05) coloured by LOPO fold-consensus", fontsize=9)
ax.tick_params(labelsize=8)
[s.set_visible(False) for s in (ax.spines["top"], ax.spines["right"])]
plt.tight_layout()
plt.savefig(ALLDIR + "nb_volcano_by_LOPOfolds.png", dpi=150, facecolor="white")
plt.show()
tab = sig.sort_values(["lopo_folds", "adj_pvalue"], ascending=[False, True])[
    ["symbol", "gene_id", "dpsi", "adj_pvalue", "lopo_folds"]
]
print(tab.to_string(index=False))
print(
    "\nnever-selected (lopo_folds==0) significant genes:",
    list(sig.loc[sig.lopo_folds == 0, "symbol"]),
)

## Summary

- Within-family DRIMSeq at the LOPO filter: 20 genes FDR<0.05 / 33 FDR<0.1 (vs 8 / 20 lenient),
  predominantly 3'UTR-lengthening (31/33), immune/myeloid + RBP/poly(A) + GTPase/cytoskeleton.
- Panel saved as `fig_s5h_within_family_drimseq` (SVG + PNG) under `3utr/paper_2026_ASD/Fig5/supp_8/`;
  the lenient-filter version is archived alongside
  (`fig_s5h_within_family_drimseq__lenient8_ARCHIVE/`).
- Exploratory volcano (`nb_volcano_by_LOPOfolds.png`) colours the significant genes by LOPO fold-
  consensus for the p-value-vs-classifier-selection view.
